In [96]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [97]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


In [98]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
df = train_df.copy()
df_test = test_df.copy()
train_df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


EDA Analysis

In [99]:
# def clean_data(df):
#     cleaned_df = df.copy()
#     cleaned_df['Age'] = cleaned_df['Age'].fillna(cleaned_df['Age'].mean())
#     cleaned_df['Sex'] = cleaned_df['Sex'].map({'male':0, 'female':1}, inplace=True)
#     cleaned_df.drop(['Name', 'Ticket', 'Cabin'], axis=1, inplace=True)
#     cleaned_df['Embarked'] = cleaned_df['Embarked'].map({'S':0, 'C':1, 'Q':2}, inplace=True)

#     return cleaned_df


In [100]:
# train_clean = clean_data(df)
# test_clean = clean_data(df_test)
# train_clean.info()
# train_clean.describe()


Train the model


In [101]:
# y = train_clean['Survived']
# x = train_clean.drop('Survived', axis=1)
# X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
# model = RandomForestClassifier(random_state=42)
# model.fit(X_train, y_train)
# y_pred = model.predict(X_test)
# accuracy = accuracy_score(y_test, y_pred)
# print(f"Accuracy: {accuracy:.4f}")


Evaluate

In [102]:
# test_prediction = model.predict(test_clean)
# submission = pd.DataFrame({
#     'PassengerId': test_df['PassengerId'], 'Survived': test_prediction})
# submission.to_csv('submission.csv', index=False)

Feature Engineering
new col of famsize, and name titles cause that would be affecting the results of who is going to be saved

In [103]:
def clean_data_2(df):
    cleaned_df = df.copy()
    cleaned_df['Title'] = cleaned_df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
    cleaned_df['Title'] = cleaned_df['Title'].replace(rare_titles, 'Rare')
    #fixing french typos
    cleaned_df['Title'] = cleaned_df['Title'].replace(['Mlle', 'Ms'], 'Miss')
    cleaned_df['Title'] = cleaned_df['Title'].replace('Mme', 'Mrs')

    title_mapping = {'Mr': 1, 'Miss': 2, 'Mrs': 3, 'Master': 4, 'Rare': 5}
    cleaned_df['Title'] = cleaned_df['Title'].map(title_mapping)
    cleaned_df['Title'] = cleaned_df['Title'].fillna(0)

    cleaned_df['FamilySize'] = cleaned_df['SibSp'] + cleaned_df['Parch'] + 1
    cleaned_df['Age'] = cleaned_df['Age'].fillna(cleaned_df['Age'].mean())
    cleaned_df['Sex'] = cleaned_df['Sex'].map({'male':0, 'female':1}, inplace=True)
    cleaned_df.drop(['Name', 'SibSp', 'Parch', 'Ticket', 'Cabin'], axis=1, inplace=True)
    cleaned_df.drop('Embarked', axis=1, inplace=True)

    return cleaned_df


In [104]:
train2_clean = clean_data_2(df)
test2_clean = clean_data_2(df_test)
train2_clean.info()
train2_clean.head()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Sex          891 non-null    int64  
 4   Age          891 non-null    float64
 5   Fare         891 non-null    float64
 6   Title        891 non-null    int64  
 7   FamilySize   891 non-null    int64  
dtypes: float64(2), int64(6)
memory usage: 55.8 KB


,PassengerId,Survived,Pclass,Sex,Age,Fare,Title,FamilySize
0,1,0,3,0,22.0,7.2500,1,2
1,2,1,1,1,38.0,71.2833,3,2
2,3,1,3,1,26.0,7.9250,2,1
3,4,1,1,1,35.0,53.1000,3,2
4,5,0,3,0,35.0,8.0500,1,1


In [105]:
# y = train2_clean['Survived']
# x = train2_clean.drop('Survived', axis=1)
# X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
# model = RandomForestClassifier(random_state=42)
# model.fit(X_train, y_train)


In [106]:
# y_pred = model.predict(X_test)
# accuracy = accuracy_score(y_test, y_pred)
# print(f"Accuracy: {accuracy:.4f}")


In [107]:
# test_prediction = model.predict(test2_clean)
# submission = pd.DataFrame({
#     'PassengerId': test_df['PassengerId'], 'Survived': test_prediction})
# submission.to_csv('submission.csv', index=False)

Now moving to make last prediction and im gonna use grid selection method to hypertune ranadomforest algo 

In [108]:
y = train2_clean['Survived']
x = train2_clean.drop('Survived', axis=1)
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [109]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 8, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
rf_base = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(estimator=rf_base, param_grid=param_grid, cv=5, n_jobs=-1, scoring='accuracy')
grid_search.fit(X_train, y_train)
print(f'Best parameter:', grid_search.best_params_)


Best parameter: {'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 100}


In [110]:

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.8156


In [111]:
test_pred = best_model.predict(test2_clean)
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'], 'Survived': test_prediction})
submission.to_csv('new_submission.csv', index=False)